<a href="https://colab.research.google.com/github/Ruturaj2472/Langchain-setup-hands-on/blob/main/Mistral_7B_Text_Generation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# LangChain Chains — Mistral-7B Text Generation

This notebook sets up **Mistral-7B-v0.1** locally (via Hugging Face `transformers`, 4-bit quantized with `bitsandbytes`) and wraps it as a LangChain LLM using `HuggingFacePipeline`. It then builds a simple **LCEL chain** (`prompt | llm | output_parser`) using a `PromptTemplate` that asks the model to act as an expert and explain a given scientific process (e.g., *Photosynthesis*). The chain is invoked with a topic input, and the generated explanation is printed as output.

**Key steps:**
1. Install dependencies (`langchain`, `langchain-huggingface`, `bitsandbytes`, etc.) and authenticate with Hugging Face.
2. Load the Mistral-7B model and tokenizer with 4-bit quantization for memory-efficient inference.
3. Create a `text-generation` pipeline and wrap it with `HuggingFacePipeline`.
4. Define a `PromptTemplate` and build an LCEL chain to generate scientific-process explanations from a topic.
---



In [1]:
!pip install langchain cohere langchain-huggingface bitsandbytes langchain-classic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 357.0/357.0 kB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 20.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 26.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 52.0 MB/s eta 0:00:00


In [2]:
from google.colab import userdata
from huggingface_hub import login
login(userdata.get('HuggingFace'))

In [3]:
import torch
import transformers
from transformers import AutoTokenizer,BitsAndBytesConfig,AutoModelForCausalLM
from langchain_classic.chains import LLMChain
from langchain_core.prompts import PromptTemplate
from langchain_huggingface.llms import HuggingFacePipeline

In [4]:
model_name='mistralai/Mistral-7B-v0.1'

model_config = transformers.AutoConfig.from_pretrained(model_name)

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

use_4bit = True

bnb_4bit_compute_dtype = "float16"
bnb_4bit_quant_type = "nf4"
use_nested_quant = False

compute_dtype = getattr(torch, bnb_4bit_compute_dtype)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=use_4bit,
    bnb_4bit_quant_type=bnb_4bit_quant_type,
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_use_double_quant=use_nested_quant,
)

# Check GPU compatibility with bfloat16
if compute_dtype == torch.float16 and use_4bit:
    major, _ = torch.cuda.get_device_capability()
    if major >= 8:
        print("=" * 80)
        print("Your GPU supports bfloat16: accelerate training with bf16=True")
        print("=" * 80)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
)

text_generation_pipeline = transformers.pipeline(
    model=model,
    tokenizer=tokenizer,
    task="text-generation",
    temperature=0.1,
    max_new_tokens=512,
    output_scores=True
)

mistral_llm = HuggingFacePipeline(pipeline=text_generation_pipeline)

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/996 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.80M [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  493kB            

tokenizer.model: downloading bytes:           |  0.00B            

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

[transformers] The following generation flags are not valid and may be ignored: ['output_scores']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'temperature', 'output_scores'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


## Sequential Chain

In [10]:
from langchain_core.output_parsers import StrOutputParser

template = """
I want you to act as a expert who can generate details on the given scientific process {topic}.
"""

prompt_template = PromptTemplate(
    input_variables=["topic"],
    template=template,
)

chain = prompt_template | mistral_llm | StrOutputParser()

In [11]:
description = "Photosynthesis"
prompt_template.format(topic=description)

'\nI want you to act as a expert who can generate details on the given scientific process Photosynthesis.\n'

In [12]:
print(chain.invoke(input={'topic':description}))

[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



I want you to act as a expert who can generate details on the given scientific process Photosynthesis.

The process of photosynthesis is a process that is carried out by plants and other organisms. It is a process that is carried out by plants and other organisms. It is a process that is carried out by plants and other organisms. It is a process that is carried out by plants and other organisms. It is a process that is carried out by plants and other organisms. It is a process that is carried out by plants and other organisms. It is a process that is carried out by plants and other organisms. It is a process that is carried out by plants and other organisms. It is a process that is carried out by plants and other organisms. It is a process that is carried out by plants and other organisms. It is a process that is carried out by plants and other organisms. It is a process that is carried out by plants and other organisms. It is a process that is carried out by plants and other organism